In [1]:
from pathlib import Path
from zipfile import ZipFile, BadZipFile
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import json
import os
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

ZIP_PATH = Path("/home/rishi/Downloads/DE 3.zip")

EXTRACT_DIR = Path(
    "/home/rishi/Documents/vscode/BacktestEngine/dataextracted")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 100)

# ============================================================
# 1. ARCHIVE INSPECTION
# ============================================================

print("\n" + "=" * 80)
print("1. ARCHIVE INSPECTION")
print("=" * 80)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"ZIP file not found: {ZIP_PATH}")

with ZipFile(ZIP_PATH, "r") as archive:
    bad_file = archive.testzip()

    if bad_file:
        raise BadZipFile(f"Corrupted archive member: {bad_file}")

    members = archive.infolist()

    total_compressed = sum(f.compress_size for f in members)
    total_uncompressed = sum(f.file_size for f in members)

    print(f"Archive: {ZIP_PATH.name}")
    print(f"Total entries: {len(members)}")
    print(f"Compressed size: {total_compressed / 1024**2:.2f} MB")
    print(f"Uncompressed size: {total_uncompressed / 1024**2:.2f} MB")

    print("\nArchive contents:")
    for member in members:
        print(
            f"{member.filename:<75} "
            f"{member.file_size / 1024**2:>10.2f} MB"
        )

    # Extract only if the destination does not already exist.
    # Resolve paths to prevent ZIP path traversal.
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    root = EXTRACT_DIR.resolve()

    for member in members:
        destination = (root / member.filename).resolve()

        if destination != root and root not in destination.parents:
            raise ValueError(f"Unsafe archive path: {member.filename}")

    if not any(EXTRACT_DIR.iterdir()):
        archive.extractall(EXTRACT_DIR)
        print(f"\nExtracted to: {EXTRACT_DIR}")
    else:
        print(f"\nExtraction directory already contains files: {EXTRACT_DIR}")

# ============================================================
# 2. DIRECTORY AND FILE DISCOVERY
# ============================================================

print("\n" + "=" * 80)
print("2. EXTRACTED FILE DISCOVERY")
print("=" * 80)

all_files = [p for p in EXTRACT_DIR.rglob("*") if p.is_file()]

parquet_files = [
    p for p in all_files
    if p.suffix.lower() in [".parquet", ".pq"]
]

csv_files = [
    p for p in all_files
    if p.suffix.lower() == ".csv"
]

print(f"Total extracted files: {len(all_files)}")
print(f"Parquet files: {len(parquet_files)}")
print(f"CSV files: {len(csv_files)}")

print("\nDirectory structure:")
for path in sorted(EXTRACT_DIR.rglob("*")):
    relative = path.relative_to(EXTRACT_DIR)
    if path.is_dir():
        print(f"[DIR]  {relative}/")
    else:
        print(f"[FILE] {relative} ({path.stat().st_size / 1024**2:.2f} MB)")

# ============================================================
# 3. PARQUET METADATA AND SCHEMA ANALYSIS
# ============================================================

print("\n" + "=" * 80)
print("3. PARQUET SCHEMA ANALYSIS")
print("=" * 80)

parquet_metadata = []

for file in parquet_files:
    try:
        parquet = pq.ParquetFile(file)
        metadata = parquet.metadata
        schema = parquet.schema_arrow

        print(f"\nFILE: {file.relative_to(EXTRACT_DIR)}")
        print("-" * 80)
        print(f"Rows: {metadata.num_rows:,}")
        print(f"Columns: {metadata.num_columns}")
        print(f"Row groups: {metadata.num_row_groups}")
        print(f"Size: {file.stat().st_size / 1024**2:.2f} MB")

        print("\nSchema:")
        print(schema)

        parquet_metadata.append({
            "file": str(file.relative_to(EXTRACT_DIR)),
            "rows": metadata.num_rows,
            "columns": metadata.num_columns,
            "row_groups": metadata.num_row_groups,
            "size_mb": file.stat().st_size / 1024**2,
            "schema": str(schema)
        })

    except Exception as error:
        print(f"ERROR reading {file.name}: {error}")

# ============================================================
# 4. SAMPLE RECORDS AND DATA TYPES
# ============================================================

print("\n" + "=" * 80)
print("4. SAMPLE DATA ANALYSIS")
print("=" * 80)

for file in parquet_files:
    try:
        table = pq.read_table(file)
        df = table.slice(0, min(5, table.num_rows)).to_pandas()

        print(f"\nFILE: {file.relative_to(EXTRACT_DIR)}")
        print("\nFirst 5 records:")
        print(df.to_string(index=False))

        print("\nColumn data types:")
        print(df.dtypes.to_string())

    except Exception as error:
        print(f"ERROR: {error}")

# ============================================================
# 5. DATE COLUMN AND RANGE DETECTION
# ============================================================

print("\n" + "=" * 80)
print("5. DATE RANGE ANALYSIS")
print("=" * 80)

date_keywords = ["date", "datetime", "timestamp", "time", "day"]

for file in parquet_files:
    try:
        parquet = pq.ParquetFile(file)
        schema = parquet.schema_arrow

        date_columns = [
            field.name for field in schema
            if any(word in field.name.lower() for word in date_keywords)
            or pa.types.is_date(field.type)
            or pa.types.is_timestamp(field.type)
        ]

        if not date_columns:
            continue

        print(f"\nFILE: {file.relative_to(EXTRACT_DIR)}")

        for column in date_columns:
            try:
                dates = pq.read_table(
                    file,
                    columns=[column]
                ).column(0).to_pandas()

                parsed = pd.to_datetime(dates, errors="coerce")
                valid = parsed.dropna()

                if not valid.empty:
                    print(
                        f"{column}: "
                        f"{valid.min()} -> {valid.max()} "
                        f"| Valid: {len(valid):,} "
                        f"| Invalid/null: {len(dates) - len(valid):,}"
                    )

            except Exception as error:
                print(f"{column}: Could not parse ({error})")

    except Exception as error:
        print(f"ERROR: {error}")

# ============================================================
# 6. DATA QUALITY ANALYSIS
# ============================================================

print("\n" + "=" * 80)
print("6. DATA QUALITY ANALYSIS")
print("=" * 80)

for file in parquet_files:
    try:
        df = pq.read_table(file).to_pandas()

        print(f"\nFILE: {file.relative_to(EXTRACT_DIR)}")
        print(f"Total rows: {len(df):,}")
        print(f"Duplicate rows: {df.duplicated().sum():,}")

        print("\nMissing values:")
        missing = df.isna().sum()
        missing = missing[missing > 0]

        if missing.empty:
            print("No missing values detected.")
        else:
            print(missing.to_string())

        print("\nNumeric summary:")
        numeric = df.select_dtypes(include="number")

        if not numeric.empty:
            print(numeric.describe().to_string())

    except Exception as error:
        print(f"ERROR: {error}")

# ============================================================
# 7. CONSOLIDATED REPORT
# ============================================================

print("\n" + "=" * 80)
print("7. CONSOLIDATED REPORT")
print("=" * 80)

report = {
    "archive": str(ZIP_PATH),
    "extraction_directory": str(EXTRACT_DIR),
    "total_files": len(all_files),
    "parquet_files": len(parquet_files),
    "csv_files": len(csv_files),
    "total_parquet_rows": sum(x["rows"] for x in parquet_metadata),
    "parquet_metadata": parquet_metadata
}

report_path = EXTRACT_DIR / "inspection_report.json"

with open(report_path, "w") as f:
    json.dump(report, f, indent=4, default=str)

print(json.dumps({
    key: value for key, value in report.items()
    if key != "parquet_metadata"
}, indent=4))

print(f"\nFull report saved to: {report_path}")
print("\nINSPECTION COMPLETE")


1. ARCHIVE INSPECTION
Archive: DE 3.zip
Total entries: 1920
Compressed size: 379.97 MB
Uncompressed size: 450.54 MB

Archive contents:
DE/                                                                               0.00 MB
__MACOSX/._DE                                                                     0.00 MB
DE/nifty500_bloomberg_extraction.ipynb.ipynb                                      2.06 MB
__MACOSX/DE/._nifty500_bloomberg_extraction.ipynb.ipynb                           0.00 MB
DE/prices/                                                                        0.00 MB
__MACOSX/DE/._prices                                                              0.00 MB
DE/.DS_Store                                                                      0.01 MB
__MACOSX/DE/._.DS_Store                                                           0.00 MB
DE/validation.ipynb                                                               5.82 MB
__MACOSX/DE/._validation.ipynb                        

In [5]:
#!/usr/bin/env python3
"""
Nifty 500 archive audit.
Run from the BacktestEngine environment:
    python audit_validate.py

Outputs:
    audit_output/audit_summary.json
    audit_output/parquet_inventory.csv
    audit_output/price_file_audit.csv
    audit_output/membership_audit.csv
    audit_output/feature_audit.csv
    audit_output/fundamentals_audit.csv
    audit_output/issues.csv

This is a data-integrity and backtest-readiness audit. It does not prove that
vendor prices are corporate-action adjusted or that feature calculations are
free of look-ahead bias; those require source/vendor evidence or recomputation.
"""

from __future__ import annotations

import json
import re
import traceback
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import numpy as np
import pyarrow.parquet as pq


# ---------------------------------------------------------------------------
# CONFIGURATION — change only ROOT if the extracted folder is elsewhere.
# The uploaded inspection report identified this extraction directory.
# ---------------------------------------------------------------------------
ROOT = Path.home() / "Documents" / "vscode" / "BacktestEngine" / "dataextracted"
OUTPUT = ROOT / "audit_output"

FEATURE_NAME = "nifty500_historical_stock_features.parquet"
MEMBERSHIP_NAME = "nifty500_historical_membership.parquet"
FUNDAMENTALS_NAME = "nifty500_historical_fundamentals.parquet"
CURRENT_FUNDAMENTALS_NAME = "nifty500_fundamentals.parquet"

DATE_CANDIDATES = ["Date", "date", "DATE"]
TICKER_CANDIDATES = ["Ticker", "ticker", "SYMBOL", "Symbol"]
PRICE_CANDIDATES = ["PX_LAST", "Close", "close", "Adj Close", "Adj_Close"]
OPEN_CANDIDATES = ["PX_OPEN", "Open", "open"]
HIGH_CANDIDATES = ["PX_HIGH", "High", "high"]
LOW_CANDIDATES = ["PX_LOW", "Low", "low"]
VOLUME_CANDIDATES = ["PX_VOLUME", "Volume", "volume"]


def find_col(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


def norm_ticker(value):
    if pd.isna(value):
        return None
    return str(value).strip().upper()


def json_default(obj):
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if hasattr(obj, "item"):
        return obj.item()
    return str(obj)


def add_issue(issues, severity, check, file="", detail=""):
    issues.append({
        "severity": severity,
        "check": check,
        "file": str(file),
        "detail": str(detail)[:1500],
    })


def locate_one(root, filename):
    matches = list(root.rglob(filename))
    return matches[0] if matches else None


def inspect_parquet_inventory(root, issues):
    rows = []
    for path in root.rglob("*.parquet"):
        try:
            pf = pq.ParquetFile(path)
            schema = pf.schema_arrow
            rows.append({
                "file": str(path.relative_to(root)),
                "rows": pf.metadata.num_rows,
                "row_groups": pf.metadata.num_row_groups,
                "columns": len(schema.names),
                "schema": " | ".join(
                    f"{field.name}:{field.type}" for field in schema
                ),
                "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
                "readable_metadata": True,
            })
        except Exception as exc:
            rows.append({
                "file": str(path.relative_to(root)),
                "rows": None, "row_groups": None, "columns": None,
                "schema": "", "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
                "readable_metadata": False,
            })
            add_issue(issues, "CRITICAL", "parquet_metadata_unreadable", path, exc)
    return pd.DataFrame(rows)


def audit_membership(path, issues):
    if path is None:
        add_issue(issues, "CRITICAL", "membership_file_missing", "", MEMBERSHIP_NAME)
        return None, {}
    df = pd.read_parquet(path)
    date_col = find_col(df.columns, DATE_CANDIDATES)
    ticker_col = find_col(df.columns, TICKER_CANDIDATES)
    if not date_col or not ticker_col:
        add_issue(issues, "CRITICAL", "membership_schema_invalid", path,
                  f"Columns found: {list(df.columns)}")
        return None, {}

    df = df[[date_col, ticker_col]].copy()
    df.columns = ["Date", "Ticker"]
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Ticker"] = df["Ticker"].map(norm_ticker)

    invalid = df["Date"].isna() | df["Ticker"].isna() | df["Ticker"].eq("")
    if invalid.any():
        add_issue(issues, "HIGH", "membership_invalid_keys", path,
                  f"{int(invalid.sum())} rows have invalid date/ticker")

    valid = df.loc[~invalid].copy()
    duplicate_count = int(valid.duplicated(["Date", "Ticker"]).sum())
    if duplicate_count:
        add_issue(issues, "HIGH", "membership_duplicate_date_ticker", path,
                  f"{duplicate_count} duplicate Date/Ticker rows")

    per_date = valid.groupby("Date")["Ticker"].agg(
        rows="size", unique_tickers="nunique"
    ).reset_index()
    per_date["duplicate_tickers"] = per_date["rows"] - per_date["unique_tickers"]
    odd = per_date[(per_date["unique_tickers"] < 450) |
                   (per_date["unique_tickers"] > 550)]
    if len(odd):
        add_issue(issues, "MEDIUM", "membership_constituent_count_outlier", path,
                  f"{len(odd)} membership dates have fewer than 450 or more than 550 unique tickers")

    stats = {
        "rows": int(len(df)),
        "valid_rows": int(len(valid)),
        "unique_tickers": int(valid["Ticker"].nunique()),
        "unique_dates": int(valid["Date"].nunique()),
        "min_date": valid["Date"].min() if len(valid) else None,
        "max_date": valid["Date"].max() if len(valid) else None,
        "duplicate_date_ticker_rows": duplicate_count,
        "invalid_key_rows": int(invalid.sum()),
        "membership_dates_with_450_to_550_tickers": int(
            per_date["unique_tickers"].between(450, 550).sum()
        ),
        "membership_date_count_outliers": int(len(odd)),
    }
    per_ticker = valid.groupby("Ticker")["Date"].agg(
        membership_first="min", membership_last="max", membership_rows="size"
    ).reset_index()
    return valid, {"stats": stats, "per_date": per_date, "per_ticker": per_ticker}


def audit_price_files(root, membership, issues):
    candidates = []
    for path in root.rglob("*.parquet"):
        try:
            cols = pq.ParquetFile(path).schema_arrow.names
        except Exception:
            continue
        if find_col(cols, DATE_CANDIDATES) and find_col(cols, PRICE_CANDIDATES):
            # Exclude consolidated feature/fundamental tables from raw-price scan.
            lowpath = str(path).lower()
            if "feature" in lowpath or "fundamental" in lowpath:
                continue
            candidates.append(path)

    membership_tickers = set(membership["Ticker"].unique()) if membership is not None else set()
    records = []
    all_price_tickers = set()
    for path in candidates:
        try:
            pf = pq.ParquetFile(path)
            cols = pf.schema_arrow.names
            date_c = find_col(cols, DATE_CANDIDATES)
            ticker_c = find_col(cols, TICKER_CANDIDATES)
            price_c = find_col(cols, PRICE_CANDIDATES)
            open_c = find_col(cols, OPEN_CANDIDATES)
            high_c = find_col(cols, HIGH_CANDIDATES)
            low_c = find_col(cols, LOW_CANDIDATES)
            vol_c = find_col(cols, VOLUME_CANDIDATES)
            wanted = list(dict.fromkeys(
                [c for c in [date_c, ticker_c, price_c, open_c, high_c, low_c, vol_c] if c]
            ))
            df = pd.read_parquet(path, columns=wanted)
            df[date_c] = pd.to_datetime(df[date_c], errors="coerce")
            if ticker_c:
                df["_ticker"] = df[ticker_c].map(norm_ticker)
            else:
                # Individual price files commonly encode the ticker in the filename.
                df["_ticker"] = path.stem.upper()
            df["_price"] = pd.to_numeric(df[price_c], errors="coerce")
            valid_tickers = set(df["_ticker"].dropna().unique())
            all_price_tickers.update(valid_tickers)

            duplicate_rows = int(df.duplicated([date_c, "_ticker"]).sum())
            invalid_dates = int(df[date_c].isna().sum())
            invalid_prices = int((df["_price"].isna() | (df["_price"] <= 0)).sum())
            null_tickers = int(df["_ticker"].isna().sum())
            bad_ohlc = 0
            if all(c is not None for c in [open_c, high_c, low_c]):
                o = pd.to_numeric(df[open_c], errors="coerce")
                h = pd.to_numeric(df[high_c], errors="coerce")
                l = pd.to_numeric(df[low_c], errors="coerce")
                c = pd.to_numeric(df[price_c], errors="coerce")
                bad_ohlc = int(((h < l) | (h < o) | (h < c) |
                                (l > o) | (l > c)).fillna(False).sum())

            # Large calendar gaps are candidates for missing sessions. Weekends
            # and exchange holidays mean this is a flag, not proof of missing data.
            gap_count = 0
            unsorted_rows = 0
            for _, ticker_frame in df.dropna(subset=[date_c, "_ticker"]).groupby("_ticker"):
                ordered = ticker_frame[date_c].sort_values()
                unsorted_rows += int((ticker_frame[date_c].reset_index(drop=True) !=
                                      ordered.reset_index(drop=True)).sum())
                gaps = ordered.diff().dt.days
                gap_count += int((gaps > 7).sum())
            negative_volume = 0
            if vol_c:
                v = pd.to_numeric(df[vol_c], errors="coerce")
                negative_volume = int((v < 0).sum())

            clean_dates = df[date_c].dropna()
            ticker = next(iter(valid_tickers)) if len(valid_tickers) == 1 else (
                ",".join(sorted(valid_tickers)[:5]) if valid_tickers else path.stem.upper()
            )
            in_membership = bool(valid_tickers & membership_tickers) if membership_tickers else None
            not_in_membership = sorted(valid_tickers - membership_tickers) if membership_tickers else []
            rec = {
                "file": str(path.relative_to(root)),
                "rows": int(len(df)),
                "ticker_or_tickers": ticker,
                "unique_tickers_in_file": len(valid_tickers),
                "first_date": clean_dates.min() if len(clean_dates) else None,
                "last_date": clean_dates.max() if len(clean_dates) else None,
                "duplicate_date_ticker_rows": duplicate_rows,
                "invalid_dates": invalid_dates,
                "invalid_or_nonpositive_close": invalid_prices,
                "null_tickers": null_tickers,
                "bad_ohlc_rows": bad_ohlc,
                "negative_volume_rows": negative_volume,
                "calendar_gaps_over_7_days": gap_count,
                "unsorted_date_rows": unsorted_rows,
                "ticker_overlaps_membership": in_membership,
                "tickers_not_in_membership": ";".join(not_in_membership[:30]),
            }
            records.append(rec)

            for key, count, sev in [
                ("price_duplicate_date_ticker", duplicate_rows, "HIGH"),
                ("price_invalid_dates", invalid_dates, "HIGH"),
                ("price_invalid_or_nonpositive_close", invalid_prices, "HIGH"),
                ("price_null_tickers", null_tickers, "HIGH"),
                ("price_bad_ohlc", bad_ohlc, "HIGH"),
                ("price_negative_volume", negative_volume, "MEDIUM"),
                ("price_calendar_gaps_over_7_days", gap_count, "MEDIUM"),
                ("price_unsorted_date_rows", unsorted_rows, "LOW"),
            ]:
                if count:
                    add_issue(issues, sev, key, path, f"{count} affected rows")
            if membership_tickers and not_in_membership:
                add_issue(issues, "MEDIUM", "price_tickers_absent_from_membership", path,
                          f"Ticker(s) absent from membership file: {not_in_membership[:30]}")
        except Exception as exc:
            add_issue(issues, "CRITICAL", "price_file_read_failed", path, exc)
            records.append({"file": str(path.relative_to(root)), "read_error": str(exc)})

    return pd.DataFrame(records), all_price_tickers, len(candidates)


def audit_features(path, membership, issues):
    if path is None:
        add_issue(issues, "HIGH", "feature_dataset_missing", "", FEATURE_NAME)
        return {}, pd.DataFrame()
    try:
        pf = pq.ParquetFile(path)
        cols = pf.schema_arrow.names
        date_c = find_col(cols, DATE_CANDIDATES)
        ticker_c = find_col(cols, TICKER_CANDIDATES)
        if not date_c or not ticker_c:
            add_issue(issues, "CRITICAL", "feature_schema_invalid", path, str(cols))
            return {}, pd.DataFrame()
        df = pd.read_parquet(path)
        df[date_c] = pd.to_datetime(df[date_c], errors="coerce")
        df[ticker_c] = df[ticker_c].map(norm_ticker)
        dup = int(df.duplicated([date_c, ticker_c]).sum())
        invalid = int((df[date_c].isna() | df[ticker_c].isna()).sum())
        feature_cols = [c for c in df.columns if c not in [date_c, ticker_c]]
        numeric_cols = df[feature_cols].select_dtypes(include="number").columns.tolist()
        inf_counts = {
            c: int((~np.isfinite(pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float,
                                                                                na_value=np.nan))).sum())
            for c in numeric_cols
        }
        inf_counts = {k: v for k, v in inf_counts.items() if v}
        if dup:
            add_issue(issues, "HIGH", "feature_duplicate_date_ticker", path, f"{dup} duplicate keys")
        if invalid:
            add_issue(issues, "HIGH", "feature_invalid_keys", path, f"{invalid} invalid keys")
        if inf_counts:
            add_issue(issues, "MEDIUM", "feature_infinite_or_non_numeric_values", path, str(inf_counts))

        membership_keys = None
        if membership is not None:
            membership_keys = set(zip(membership["Date"], membership["Ticker"]))
            feature_keys = set(zip(df[date_c], df[ticker_c]))
            # Exact key comparison is valid only if membership is a daily snapshot.
            # The report alone does not establish that semantic, so this is diagnostic.
            feature_not_membership = len(feature_keys - membership_keys)
            membership_not_feature = len(membership_keys - feature_keys)
        else:
            feature_not_membership = membership_not_feature = None

        stats = {
            "rows": int(len(df)),
            "columns": int(len(df.columns)),
            "unique_tickers": int(df[ticker_c].nunique()),
            "unique_dates": int(df[date_c].nunique()),
            "first_date": df[date_c].min(),
            "last_date": df[date_c].max(),
            "duplicate_date_ticker_rows": dup,
            "invalid_key_rows": invalid,
            "numeric_feature_columns": numeric_cols,
            "feature_missing_counts": {
                c: int(df[c].isna().sum()) for c in numeric_cols if df[c].isna().any()
            },
            "feature_nonfinite_counts": inf_counts,
            "feature_keys_not_exactly_in_membership": feature_not_membership,
            "membership_keys_not_exactly_in_features": membership_not_feature,
            "exact_membership_key_comparison_warning":
                "Diagnostic only: membership may be event-based rather than a daily snapshot.",
        }
        return stats, pd.DataFrame([{
            "feature": c,
            "missing": int(df[c].isna().sum()),
            "missing_pct": round(float(df[c].isna().mean() * 100), 4),
            "unique_values": int(df[c].nunique(dropna=True)),
            "min": pd.to_numeric(df[c], errors="coerce").min(),
            "max": pd.to_numeric(df[c], errors="coerce").max(),
        } for c in numeric_cols])
    except Exception as exc:
        add_issue(issues, "CRITICAL", "feature_dataset_read_failed", path, exc)
        return {"error": str(exc)}, pd.DataFrame()


def audit_fundamentals(path, issues):
    if path is None:
        add_issue(issues, "MEDIUM", "historical_fundamentals_missing", "", FUNDAMENTALS_NAME)
        return {}
    try:
        df = pd.read_parquet(path)
        date_c = find_col(df.columns, DATE_CANDIDATES)
        security_c = find_col(df.columns, ["Bloomberg_Security", "Ticker", "ticker"])
        if not date_c or not security_c:
            add_issue(issues, "HIGH", "fundamentals_schema_invalid", path, str(list(df.columns)))
            return {"columns": list(df.columns), "rows": len(df)}
        df[date_c] = pd.to_datetime(df[date_c], errors="coerce")
        df[security_c] = df[security_c].astype("string").str.strip()
        dup = int(df.duplicated([date_c, security_c]).sum())
        if dup:
            add_issue(issues, "HIGH", "fundamentals_duplicate_security_date", path,
                      f"{dup} duplicate security/date rows")
        numeric = df.select_dtypes(include="number").columns
        missing = {c: int(df[c].isna().sum()) for c in numeric if df[c].isna().any()}
        return {
            "rows": int(len(df)),
            "columns": list(df.columns),
            "unique_securities": int(df[security_c].nunique()),
            "unique_dates": int(df[date_c].nunique()),
            "first_date": df[date_c].min(),
            "last_date": df[date_c].max(),
            "invalid_dates": int(df[date_c].isna().sum()),
            "duplicate_security_date_rows": dup,
            "numeric_missing_counts": missing,
            "publication_timestamp_present": any(
                x in df.columns for x in ["Publication_Date", "Announcement_Date",
                                           "Filing_Date", "As_Of_Date", "Timestamp"]
            ),
            "point_in_time_warning":
                "A Date column is not proof of when the information became public. "
                "Publication/announcement timestamps are needed to prevent fundamental look-ahead.",
        }
    except Exception as exc:
        add_issue(issues, "CRITICAL", "fundamentals_read_failed", path, exc)
        return {"error": str(exc)}


def inspect_failure_logs(root, issues):
    logs = []
    for path in root.rglob("*.csv"):
        name = path.name.lower()
        if any(token in name for token in ["fail", "error", "audit", "missing"]):
            try:
                df = pd.read_csv(path)
                logs.append({
                    "file": str(path.relative_to(root)),
                    "rows": int(len(df)),
                    "columns": list(df.columns),
                    "sample": df.head(5).astype(str).to_dict(orient="records"),
                })
                if len(df):
                    add_issue(issues, "MEDIUM", "nonempty_failure_or_missing_data_log",
                              path, f"{len(df)} rows; inspect this log before accepting the dataset")
            except Exception as exc:
                add_issue(issues, "HIGH", "failure_log_read_failed", path, exc)
    return logs


def main():
    if not ROOT.exists():
        raise SystemExit(
            f"Extraction directory not found: {ROOT}\n"
            "Edit ROOT at the top of this script to point to the folder containing DE/."
        )
    OUTPUT.mkdir(parents=True, exist_ok=True)
    issues = []

    parquet_files = list(ROOT.rglob("*.parquet"))
    csv_files = list(ROOT.rglob("*.csv"))
    inventory = inspect_parquet_inventory(ROOT, issues)
    if not inventory.empty:
        inventory.to_csv(OUTPUT / "parquet_inventory.csv", index=False)

    membership_path = locate_one(ROOT, MEMBERSHIP_NAME)
    membership, membership_info = audit_membership(membership_path, issues)

    price_audit, price_tickers, price_file_count = audit_price_files(ROOT, membership, issues)
    if not price_audit.empty:
        price_audit.to_csv(OUTPUT / "price_file_audit.csv", index=False)

    feature_path = locate_one(ROOT, FEATURE_NAME)
    feature_stats, feature_table = audit_features(feature_path, membership, issues)
    if not feature_table.empty:
        feature_table.to_csv(OUTPUT / "feature_audit.csv", index=False)

    fundamentals_path = locate_one(ROOT, FUNDAMENTALS_NAME)
    fundamentals_stats = audit_fundamentals(fundamentals_path, issues)

    current_fund_path = locate_one(ROOT, CURRENT_FUNDAMENTALS_NAME)
    current_fund_rows = None
    if current_fund_path:
        try:
            current_fund_rows = int(pq.ParquetFile(current_fund_path).metadata.num_rows)
            if current_fund_rows == 0:
                add_issue(issues, "MEDIUM", "empty_current_fundamentals_file",
                          current_fund_path,
                          "File has zero rows. Determine whether it is intentionally unused or a failed extraction.")
        except Exception as exc:
            add_issue(issues, "HIGH", "current_fundamentals_metadata_failed", current_fund_path, exc)

    failure_logs = inspect_failure_logs(ROOT, issues)

    # Cross-dataset coverage summaries.
    membership_tickers = set(membership["Ticker"].unique()) if membership is not None else set()
    missing_price_tickers = sorted(membership_tickers - price_tickers) if membership_tickers else []
    extra_price_tickers = sorted(price_tickers - membership_tickers) if membership_tickers else []
    if missing_price_tickers:
        add_issue(issues, "HIGH", "membership_tickers_without_price_file", "",
                  f"{len(missing_price_tickers)} tickers; sample={missing_price_tickers[:50]}")
    if extra_price_tickers:
        add_issue(issues, "MEDIUM", "price_tickers_not_in_membership", "",
                  f"{len(extra_price_tickers)} tickers; sample={extra_price_tickers[:50]}")

    # Explicitly flag what cannot be validated from the supplied archive/report alone.
    corp_action_files = [
        p for p in ROOT.rglob("*")
        if p.is_file() and any(s in p.name.lower() for s in
                               ["corporate_action", "corporateaction", "split", "bonus", "dividend"])
    ]
    if not corp_action_files:
        add_issue(issues, "BLOCKER", "corporate_action_adjustment_not_verifiable", "",
                  "No identifiable corporate-action dataset found. Cannot establish split/bonus/dividend adjustment correctness.")
    add_issue(issues, "BLOCKER", "feature_lookahead_not_proven", feature_path or "",
              "Feature calculations were not recomputed from raw data with a strict as-of cutoff. "
              "No-look-ahead status is unproven.")
    add_issue(issues, "BLOCKER", "fundamental_point_in_time_not_proven", fundamentals_path or "",
              "Fundamental dates are not sufficient unless they represent public-availability dates or are joined using publication timestamps.")

    severity_counts = Counter(x["severity"] for x in issues)
    summary = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "root": str(ROOT),
        "output": str(OUTPUT),
        "archive_inventory": {
            "parquet_files": len(parquet_files),
            "csv_files": len(csv_files),
            "parquet_metadata_readable": int(inventory["readable_metadata"].sum()) if not inventory.empty else 0,
            "parquet_metadata_unreadable": int((~inventory["readable_metadata"]).sum()) if not inventory.empty else 0,
            "parquet_rows_from_metadata": int(inventory["rows"].fillna(0).sum()) if not inventory.empty else 0,
        },
        "membership": membership_info.get("stats") if membership_info else None,
        "prices": {
            "price_like_parquet_files": price_file_count,
            "unique_price_tickers": len(price_tickers),
            "membership_tickers_without_price_file_count": len(missing_price_tickers),
            "membership_tickers_without_price_file_sample": missing_price_tickers[:100],
            "price_tickers_not_in_membership_count": len(extra_price_tickers),
            "price_tickers_not_in_membership_sample": extra_price_tickers[:100],
        },
        "features": feature_stats,
        "historical_fundamentals": fundamentals_stats,
        "empty/current_fundamentals_rows": current_fund_rows,
        "failure_logs": failure_logs,
        "corporate_action_files_found": [str(p.relative_to(ROOT)) for p in corp_action_files],
        "issue_counts": dict(severity_counts),
        "total_issues": len(issues),
        "readiness_assessment": (
            "NOT BACKTEST-READY until blocker issues are resolved. "
            "Passing structural checks does not establish point-in-time correctness."
        ),
    }

    pd.DataFrame(issues).to_csv(OUTPUT / "issues.csv", index=False)
    with open(OUTPUT / "audit_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, default=json_default)

    print("\nAUDIT COMPLETE")
    print(f"Root: {ROOT}")
    print(f"Parquet files: {len(parquet_files)}")
    print(f"Price-like Parquet files: {price_file_count}")
    print(f"Membership tickers: {len(membership_tickers)}")
    print(f"Price tickers: {len(price_tickers)}")
    print(f"Issues: {len(issues)} | {dict(severity_counts)}")
    print(f"Output: {OUTPUT}")
    print("Read audit_output/audit_summary.json and issues.csv first.")


if __name__ == "__main__":
    main()



AUDIT COMPLETE
Root: /home/rishi/Documents/vscode/BacktestEngine/dataextracted
Parquet files: 1754
Price-like Parquet files: 860
Membership tickers: 861
Price tickers: 858
Issues: 1811 | {'CRITICAL': 877, 'MEDIUM': 920, 'HIGH': 12, 'BLOCKER': 2}
Output: /home/rishi/Documents/vscode/BacktestEngine/dataextracted/audit_output
Read audit_output/audit_summary.json and issues.csv first.
